## Лабораторна робота №2. Частина 2
**Завдання:** Завантаження великого датасету Power Consumption, аналіз швидкості за допомогою `timeit`, нормалізація даних та розрахунок коефіцієнтів кореляції.


In [1]:
import os
import zipfile
import urllib.request
import timeit
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

# Автоматичне завантаження офіційного датасету UCI
url = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"
if not os.path.exists("household_power_consumption.txt"):
    print("Завантаження архіву даних (це може зайняти до хвилини)...")
    urllib.request.urlretrieve(url, "power.zip")
    with zipfile.ZipFile("power.zip", 'r') as z:
        z.extractall(".")
    print("Архів успішно розархівовано.")
        
# Читаємо дані
df_p = pd.read_csv("household_power_consumption.txt", sep=";", low_memory=False)
df_p.replace('?', np.nan, inplace=True)
df_p.dropna(inplace=True)

# Перетворюємо технічні колонки на числа
for c in ["Global_active_power", "Global_reactive_power", "Voltage", "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]:
    df_p[c] = pd.to_numeric(df_p[c])

print("✅ Дані очищено та підготовлено! Розмір таблиці:", df_p.shape)

Завантаження архіву даних (це може зайняти до хвилини)...
Архів успішно розархівовано.
✅ Дані очищено та підготовлено! Розмір таблиці: (2049280, 9)


In [3]:
print(" Початок замірів швидкості виконання операцій за допомогою timeit...\n")

# 1. Потужність > 5 кВт
t1 = timeit.timeit(lambda: df_p[df_p["Global_active_power"] > 5.0], number=1)
print(f"🔸 Процедура 1 (Фільтрація > 5кВт) зайняла: {t1:.4f} сек")

# 2. Вольтаж > 235 В та струм 19-20 А
def proc2():
    return df_p[(df_p["Voltage"] > 235.0) & (df_p["Global_intensity"] >= 19.0) & (df_p["Global_intensity"] <= 20.0)]
t2 = timeit.timeit(proc2, number=1)
print(f"🔸 Процедура 2 (Складний фільтр струму) зайняла: {t2:.4f} сек")

# 3. Випадковий вибір 500 000 рядків та розрахунок середнього
def proc3():
    sub_sample = df_p.sample(n=500000, replace=False)
    return sub_sample[["Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]].mean()
t3 = timeit.timeit(proc3, number=1)
print(f"🔸 Процедура 3 (Вибірка 500k рядків + Mean) зайняла: {t3:.4f} сек")

 Початок замірів швидкості виконання операцій за допомогою timeit...

🔸 Процедура 1 (Фільтрація > 5кВт) зайняла: 0.0083 сек
🔸 Процедура 2 (Складний фільтр струму) зайняла: 0.0198 сек
🔸 Процедура 3 (Вибірка 500k рядків + Mean) зайняла: 0.2799 сек


In [2]:
# Нормалізація колонки Voltage (приведення до діапазону від 0 до 1)
v_min = df_p["Voltage"].min()
v_max = df_p["Voltage"].max()
df_p["Voltage_Normalized"] = (df_p["Voltage"] - v_min) / (v_max - v_min)
print("📊 Перші 3 рядки з нормалізованим Вольтажем:")
print(df_p[["Voltage", "Voltage_Normalized"]].head(3))
print("-" * 50)

# Розрахунок коефіцієнтів кореляції Пірсона та Спірмена
p_coeff, _ = pearsonr(df_p["Global_active_power"], df_p["Global_intensity"])
s_coeff, _ = spearmanr(df_p["Global_active_power"], df_p["Global_intensity"])

print(f"📈 Коефіцієнт кореляції Пірсона: {p_coeff:.4f}")
print(f"📉 Коефіцієнт кореляції Спірмена: {s_coeff:.4f}")

📊 Перші 3 рядки з нормалізованим Вольтажем:
   Voltage  Voltage_Normalized
0   234.84            0.376090
1   233.63            0.336995
2   233.29            0.326010
--------------------------------------------------
📈 Коефіцієнт кореляції Пірсона: 0.9989
📉 Коефіцієнт кореляції Спірмена: 0.9954


In [4]:
# 1. Стандартизація (Приведення до середнього 0 та відхилення 1)
df_p["Voltage_Standardized"] = (df_p["Voltage"] - df_p["Voltage"].mean()) / df_p["Voltage"].std()

# 2. Створюємо фіктивний категоріальний атрибут для демонстрації One-Hot Encoding
# (Наприклад, розділимо день на 'Day' та 'Night')
df_p["Time_Category"] = np.where(df_p["Time"].str.split(":").str[0].astype(int) >= 18, "Night", "Day")

# 3. Проводимо One Hot Encoding за допомогою pandas
df_ohe = pd.get_dummies(df_p["Time_Category"], prefix="Period", dtype=int)
print("📊 Результат One Hot Encoding (Категоріальний атрибут):")
print(df_ohe.head(3))

📊 Результат One Hot Encoding (Категоріальний атрибут):
   Period_Day  Period_Night
0           1             0
1           1             0
2           1             0
